# nlp-getting-started — baseline notebook

Generated by `python -m engine init`. This is the **only source artifact** for
the competition (rule R1) and must run top-to-bottom on a fresh machine (R2).

Pipeline: load → target detection → feature prep → 5-fold CV →
out-of-fold metric → fold-averaged test predictions → `../train/metrics.json`
and `../train/submission.csv`.

The engine's coder may extend this notebook. It must keep the artifact
contract (R4/R13) and the leakage discipline (R6) intact.


In [ ]:
# ==== ENGINE CONFIG =========================================================
SLUG = "nlp-getting-started"
TARGET = 'target'          # None -> auto-detect from the submission template
METRIC = 'f1'          # None -> roc_auc / balanced_accuracy / rmse by task
SEED = 42
N_SPLITS = 5
DATASETS_DIR = "../datasets"     # relative to this notebook's folder (R1)
TRAIN_DIR = "../train"
TARGET_SCORE = 0.75   # score-matching gate (R5); None if not set
# ============================================================================


In [ ]:
import json, os, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, log_loss,
                             mean_absolute_error, mean_squared_error, roc_auc_score)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)
STARTED = time.time()
DATASETS_DIR, TRAIN_DIR = Path(DATASETS_DIR), Path(TRAIN_DIR)   # strings -> Paths
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
print(f"python {os.sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")


In [ ]:
def first_match(patterns):
    for pattern in patterns:
        hits = sorted(DATASETS_DIR.glob(pattern))
        if hits:
            return hits[0]
    return None

train_path = first_match(["train*.csv"])
test_path = first_match(["test*.csv"])
sub_path = first_match(["sample_submission*.csv"])

if train_path is None:
    raise FileNotFoundError(
        f"No training CSV in {DATASETS_DIR.resolve()}. "
        f"Run: python -m engine scout --slug {SLUG}"
    )

train = pd.read_csv(train_path)
test = pd.read_csv(test_path) if test_path else None
sample_sub = pd.read_csv(sub_path) if sub_path else None
print("train:", train.shape,
      "| test:", None if test is None else test.shape,
      "| sample_submission:", None if sample_sub is None else sample_sub.shape)


In [ ]:
id_cols = [c for c in train.columns
           if c.strip().lower() in {"id", "index", "row_id", "rowid"}
           or c.strip().lower().endswith("_id")]
sub_cols = list(sample_sub.columns) if sample_sub is not None else None

if TARGET is None:
    candidates = [c for c in train.columns if c not in (sub_cols or [])]
    candidates = [c for c in candidates if c not in id_cols]
    if len(candidates) == 1:
        TARGET = candidates[0]
    elif "target" in train.columns:
        TARGET = "target"
    else:
        TARGET = train.columns[-1]

assert TARGET in train.columns, f"TARGET={TARGET!r} not in {list(train.columns)}"
y_raw = train[TARGET].copy()

id_col = id_cols[0] if id_cols else None
train_ids = train[id_col].reset_index(drop=True) if id_col else pd.Series(np.arange(len(train)))
if test is not None:
    test_ids = test[id_col].reset_index(drop=True) if id_col in test.columns else pd.Series(np.arange(len(test)))
else:
    test_ids = None

train = train.drop(columns=[TARGET])
print("target:", TARGET, "| id column:", id_col)


In [ ]:
n_unique = y_raw.nunique(dropna=True)
integer_like = pd.api.types.is_integer_dtype(y_raw) or pd.api.types.is_bool_dtype(y_raw)

if n_unique == 2 and integer_like:
    TASK = "binary"
elif not pd.api.types.is_float_dtype(y_raw) and 2 < n_unique <= 30:
    TASK = "multiclass"
else:
    TASK = "regression"

if TASK == "multiclass":
    encoder = LabelEncoder()
    y = encoder.fit_transform(y_raw.astype(str))
    print("classes:", list(encoder.classes_))
else:
    encoder = None
    y = y_raw.to_numpy()

if METRIC is None:
    METRIC = {"binary": "roc_auc", "multiclass": "balanced_accuracy", "regression": "rmse"}[TASK]
print("task:", TASK, "| metric:", METRIC)


In [ ]:
if test is None:
    raise FileNotFoundError("test CSV missing — a submission cannot be produced without it.")

feature_cols = [c for c in train.columns if c in test.columns]
dropped = [c for c in train.columns if c not in test.columns]
if dropped:
    print("dropped (not in test):", dropped)

X = train[feature_cols].copy()
X_test = test[feature_cols].copy()

cat_features = []
for col in feature_cols:
    series = X[col]
    if series.dtype == "object" or str(series.dtype) == "category":
        cat_features.append(col)
    elif pd.api.types.is_integer_dtype(series) and series.nunique() <= 25:
        cat_features.append(col)  # small integer codes behave like categories

for col in cat_features:
    combined = pd.concat([X[col], X_test[col]], axis=0)
    combined = combined.astype("string").fillna("__MISSING__")
    categories = pd.Categorical(combined).categories
    X[col] = pd.Categorical(X[col].astype("string").fillna("__MISSING__"), categories=categories)
    X_test[col] = pd.Categorical(X_test[col].astype("string").fillna("__MISSING__"), categories=categories)

num_features = [c for c in feature_cols if c not in cat_features]
for col in num_features:
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

print(f"{len(num_features)} numeric / {len(cat_features)} categorical features")


In [ ]:
def score_signal(signal, y_true):
    """Metric value for one fold, from the raw model signal."""
    if TASK == "regression":
        if METRIC == "rmse":
            return float(np.sqrt(mean_squared_error(y_true, signal)))
        if METRIC == "mse":
            return float(mean_squared_error(y_true, signal))
        if METRIC == "mae":
            return float(mean_absolute_error(y_true, signal))
        raise ValueError(f"unsupported regression metric: {METRIC}")

    if TASK == "binary":
        if METRIC in ("roc_auc", "auc"):
            return float(roc_auc_score(y_true, signal))
        if METRIC in ("logloss", "log_loss", "neg_log_loss"):
            return float(log_loss(y_true, signal))
        labels = (signal > 0.5).astype(int)
        fn = balanced_accuracy_score if METRIC == "balanced_accuracy" else accuracy_score
        return float(fn(y_true, labels))

    # multiclass: signal is a probability matrix
    labels = signal.argmax(axis=1)
    if METRIC == "balanced_accuracy":
        return float(balanced_accuracy_score(y_true, labels))
    if METRIC == "accuracy":
        return float(accuracy_score(y_true, labels))
    if METRIC in ("logloss", "log_loss", "neg_log_loss"):
        return float(log_loss(y_true, signal, labels=list(range(signal.shape[1]))))
    raise ValueError(f"unsupported multiclass metric: {METRIC}")

HIGHER_IS_BETTER = METRIC not in {"rmse", "mse", "mae", "logloss", "log_loss", "neg_log_loss"}


In [ ]:
import lightgbm as lgb

if TASK == "regression":
    folds = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
else:
    folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

n_classes = len(encoder.classes_) if TASK == "multiclass" else None
oof = np.zeros(len(X)) if TASK != "multiclass" else np.zeros((len(X), n_classes))
test_signal = np.zeros(len(X_test)) if TASK != "multiclass" else np.zeros((len(X_test), n_classes))
fold_scores, models = [], []

for fold, (tr_idx, va_idx) in enumerate(folds.split(X, y), start=1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    common = dict(n_estimators=5000, learning_rate=0.03, num_leaves=31,
                  min_child_samples=40, subsample=0.9, subsample_freq=1,
                  colsample_bytree=0.9, reg_lambda=1.0, random_state=SEED,
                  n_jobs=-1, verbosity=-1)
    if TASK == "binary":
        model = lgb.LGBMClassifier(objective="binary", **common)
    elif TASK == "multiclass":
        model = lgb.LGBMClassifier(objective="multiclass", num_class=n_classes, **common)
    else:
        model = lgb.LGBMRegressor(objective="regression", **common)

    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], categorical_feature=cat_features,
              callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])
    models.append(model)

    if TASK == "regression":
        va_signal = model.predict(X_va)
        te_signal = model.predict(X_test)
    else:
        va_signal = model.predict_proba(X_va)
        te_signal = model.predict_proba(X_test)
        if TASK == "binary":
            va_signal, te_signal = va_signal[:, 1], te_signal[:, 1]

    oof[va_idx] = va_signal
    test_signal += te_signal / N_SPLITS
    fold_scores.append(score_signal(va_signal, y_va))
    print(f"fold {fold}/{N_SPLITS}: {METRIC} = {fold_scores[-1]:.6f} "
          f"(best iter {model.best_iteration_})")

# R6: every row must have exactly one out-of-fold prediction.
assert not np.isnan(np.asarray(oof, dtype=float)).any(), "incomplete OOF coverage (rule R6)"


In [ ]:
cv_mean = float(np.mean(fold_scores))
cv_std = float(np.std(fold_scores))
runtime = round(time.time() - STARTED, 1)
oof_score = score_signal(oof, y)

metrics = {
    "slug": SLUG,
    "metric": METRIC,
    "task": TASK,
    "cv_mean": round(cv_mean, 6),
    "cv_std": round(cv_std, 6),
    "cv_folds": [round(s, 6) for s in fold_scores],
    "oof_score": round(float(oof_score), 6),
    "n_splits": N_SPLITS,
    "n_train": int(len(X)),
    "n_test": int(len(X_test)),
    "seed": SEED,
    "target": TARGET,
    "models": ["lightgbm-baseline"],
    "runtime_sec": runtime,
    "target_score": TARGET_SCORE,
    "higher_is_better": HIGHER_IS_BETTER,
}
(TRAIN_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

oof_frame = pd.DataFrame(oof)
oof_frame["y_true"] = y
oof_frame.to_csv(TRAIN_DIR / "oof_lgbm.csv", index=False)

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": np.mean([m.feature_importances_ for m in models], axis=0),
}).sort_values("importance", ascending=False)
importance.to_csv(TRAIN_DIR / "importance_lgbm.csv", index=False)

# R13: the trained model ships alongside the notebook — the fitted fold
# estimators exactly as they produced the OOF score above.
joblib.dump({"models": models, "feature_cols": feature_cols, "seed": SEED,
             "metric": METRIC, "task": TASK},
            TRAIN_DIR / "model.joblib")
print("wrote", (TRAIN_DIR / "model.joblib").resolve())

print()
print(f"OOF {METRIC}: {cv_mean:.6f} +/- {cv_std:.6f}  (folds: {metrics['cv_folds']})")
if TARGET_SCORE is not None:
    reached = cv_mean >= TARGET_SCORE if HIGHER_IS_BETTER else cv_mean <= TARGET_SCORE
    print(f"score gate: {'REACHED' if reached else 'NOT reached'} (target {TARGET_SCORE})")
importance.head(15)


In [ ]:
if TASK == "binary":
    submission_values = test_signal                      # positive-class probability
elif TASK == "multiclass":
    submission_values = test_signal.argmax(axis=1)
    if encoder is not None:
        submission_values = encoder.inverse_transform(submission_values.astype(int))
else:
    submission_values = test_signal

if sample_sub is not None:
    submission = sample_sub.copy()
    value_cols = [c for c in submission.columns if c not in id_cols]
    assert value_cols, "sample_submission has no value column"
    if len(value_cols) == 1:
        submission[value_cols[0]] = submission_values
    else:
        values = submission_values
        if np.ndim(values) == 1:
            values = np.column_stack([1 - values, values])
        submission[value_cols] = values
else:
    submission = pd.DataFrame({id_col or "id": test_ids, TARGET: submission_values})

submission.to_csv(TRAIN_DIR / "submission.csv", index=False)
print("wrote", (TRAIN_DIR / "submission.csv").resolve())
submission.head()


## Score matching (rule R5)

`../train/metrics.json` now holds the measured OOF score. Copy it into
`../readme.md` — never invent a number (R4).

- Gate target: `TARGET_SCORE`
- Direction: higher-is-better unless the metric is RMSE / MAE / log-loss.
- If the gate is not met, the engine feeds the gap back to the coding agent and
  the next attempt climbs the R7 ladder: domain features → model zoo →
  OOF-fitted ensemble → fold-safe target encoding → pseudo-labelling →
  bounded hyperparameter search.
